# Single gNodeB / Single UE PRB Trace

This notebook creates one gNodeB and one UE, then traces the difference between:

- `budget_prbs`: PRBs reserved/available for the slice.
- `allocated_prbs` / `used_prbs`: PRBs the scheduler actually gave to the UE.
- `demand_prbs`: estimated PRBs needed to drain the queue.
- `scheduled_bits` and `served_bits`: transmission attempt vs successful service.

So if the slice has `50` PRBs but only `10` are used, you should see `budget_prbs = 50` and `allocated_prbs = 10`.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scenario_creator import create_multignb_env

plt.rcParams.update({"figure.dpi": 125, "axes.grid": True, "grid.alpha": 0.25})

## Create One gNodeB And One UE

The gNodeB has `150` total PRBs. The slice budgets are configured as `50/50/50`, so eMBB has `50` PRBs available. The UE uses eMBB only.

The traffic schedule changes the UE demand over time:

- steps `0-9`: low demand
- steps `10-19`: medium demand
- steps `20-29`: high demand
- steps `30+`: no new traffic

In [ ]:
SEED = 123
SLICE_TYPE = "eMBB"
TOTAL_PRBS = 150
SLICE_PRB_BUDGETS = {"eMBB": 50, "URLLC": 50, "mMTC": 50}
MAX_PRBS_PER_UE = 20
N_STEPS = 45

rng = np.random.default_rng(SEED)

env = create_multignb_env(
    rng=rng,
    n=4,
    n_gnbs=1,
    gnb_configs=[{
        "id": 0,
        "x": 0.0,
        "y": 0.0,
        "coverage_radius": 500.0,
        "carrier_id": 0,
        "n_prbs": TOTAL_PRBS,
    }],
    slots_per_step=5,
    L1_level=False,
    step_dt=1e-3,
    mobility_dt=0.0,
    radio_substeps=1,
    max_episode_steps=N_STEPS + 5,
    slice_prb_budgets=SLICE_PRB_BUDGETS,
    max_prbs_per_ue=MAX_PRBS_PER_UE,
)

obs, info = env.reset(seed=SEED)

ue_id = env.add_ue(
    x=20.0,
    y=0.0,
    vx=0.0,
    vy=0.0,
    slice_type=SLICE_TYPE,
    bit_rate=200_000.0,
    packet_size_bits=400.0,
    bit_rate_schedule=[
        {"step": 10, "bit_rate": 1_000_000.0},
        {"step": 20, "bit_rate": 8_000_000.0},
        {"step": 30, "bit_rate": 0.0},
    ],
)

print("UE id:", ue_id)
print("UE serving gNB:", env.get_ue(ue_id).serving_gnb)
print("eMBB budget PRBs:", env.get_slice_prb_budget(0, SLICE_TYPE))
print("max PRBs per UE:", env.max_prbs_per_ue)

## Run And Collect Trace

In [ ]:
rows = []

for step in range(N_STEPS):
    obs, reward, terminated, truncated, info = env.step(0)
    kpi = info["slice_kpis"][(0, SLICE_TYPE)]
    ue_metrics = env.get_ue_radio_metrics(ue_id)

    rows.append({
        "step": step,
        "budget_prbs": kpi["budget_prbs"],
        "allocated_prbs": kpi["allocated_prbs"],
        "used_prbs": kpi["used_prbs"],
        "allocated_load": kpi["allocated_load"],
        "demand_prbs": kpi["demand_prbs"],
        "demand_load": kpi["demand_load"],
        "queue_bits": kpi["queue_bits"],
        "queue_pressure": kpi["queue_pressure"],
        "arrived_bits": kpi["arrived_bits"],
        "scheduled_bits": kpi["scheduled_bits"],
        "served_bits": kpi["served_bits"],
        "served_ratio": kpi["served_ratio"],
        "tx_success_ratio": kpi["tx_success_ratio"],
        "ue_allocated_prbs": ue_metrics["allocated_prbs"],
        "ue_queue": ue_metrics["queue"],
        "ue_new_bits": ue_metrics["new_bits"],
        "ue_served_bits": ue_metrics["served_bits"],
        "ue_scheduled_bits": ue_metrics["scheduled_bits"],
        "ue_rx_probability": ue_metrics["rx_probability"],
        "ue_sinr_db": ue_metrics["sinr_db"],
        "ue_offered_bit_rate": ue_metrics["offered_bit_rate"],
    })

    if terminated or truncated:
        break

trace = pd.DataFrame(rows)
display(trace.head(12))
display(trace.tail(12))

## Verify Budget Versus Used PRBs

In [ ]:
assert (trace["allocated_prbs"] == trace["used_prbs"]).all()
assert (trace["allocated_prbs"] <= trace["budget_prbs"]).all()
assert (trace["allocated_prbs"] <= MAX_PRBS_PER_UE).all()

summary = pd.DataFrame({
    "metric": [
        "slice budget PRBs",
        "max allocated PRBs",
        "min allocated PRBs",
        "steps with allocated < budget",
        "steps with allocated == 0",
    ],
    "value": [
        int(trace["budget_prbs"].iloc[0]),
        int(trace["allocated_prbs"].max()),
        int(trace["allocated_prbs"].min()),
        int((trace["allocated_prbs"] < trace["budget_prbs"]).sum()),
        int((trace["allocated_prbs"] == 0).sum()),
    ],
})
display(summary)

## Plots

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True, constrained_layout=True)

axes[0].plot(trace["step"], trace["budget_prbs"], label="budget_prbs", linewidth=2)
axes[0].plot(trace["step"], trace["allocated_prbs"], label="allocated/used_prbs", linewidth=2)
axes[0].plot(trace["step"], trace["demand_prbs"], label="demand_prbs", linewidth=2)
axes[0].axhline(MAX_PRBS_PER_UE, color="tab:red", linestyle="--", label="max_prbs_per_ue")
axes[0].set_ylabel("PRBs")
axes[0].legend(loc="upper left")

axes[1].plot(trace["step"], trace["allocated_load"], label="allocated_load")
axes[1].plot(trace["step"], trace["demand_load"], label="demand_load")
axes[1].set_ylabel("load")
axes[1].legend(loc="upper left")

axes[2].plot(trace["step"], trace["queue_bits"], label="queue_bits")
axes[2].plot(trace["step"], trace["arrived_bits"], label="arrived_bits")
axes[2].plot(trace["step"], trace["served_bits"], label="served_bits")
axes[2].set_ylabel("bits")
axes[2].legend(loc="upper left")

axes[3].plot(trace["step"], trace["ue_rx_probability"], label="rx_probability")
axes[3].plot(trace["step"], trace["tx_success_ratio"], label="tx_success_ratio")
axes[3].set_ylabel("ratio")
axes[3].set_xlabel("step")
axes[3].legend(loc="upper left")

plt.show()

## Close Environment

In [ ]:
env.close()